### Step 1: Install Required Python Libraries

**Question:**
> *What packages are required to set up an efficient parameter-efficient fine-tuning (PEFT) and supervised fine-tuning (SFT) environment for large language models[cite: 115]?*

**Explanation:**
* **`transformers` & `datasets`**: Core Hugging Face libraries for loading pre-trained models, tokenizers, and custom datasets[cite: 73, 115].
* **`peft` & `trl`**: Parameter-Efficient Fine-Tuning library (for LoRA) and Transformer Reinforcement Learning library (for `SFTTrainer`)[cite: 73, 115].
* **`accelerate` & `bitsandbytes`**: Optimizes memory and enables low-precision training (quantization)[cite: 73, 115].

In [2]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes "torchao>=0.16.0"


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 2: Verify GPU Hardware & Memory

**Question:**
> *How do we verify whether a GPU is available, and why is hardware acceleration necessary for fine-tuning LLMs[cite: 73, 115]?*

**Explanation:**
* **Hardware Acceleration**: Fine-tuning millions of neural network weights via backpropagation requires massive parallel matrix multiplication, making GPUs (such as the Tesla T4) essential[cite: 73, 100, 115].
* **Memory Check**: Inspects total VRAM availability to ensure the model and training batch sizes fit within hardware limits[cite: 73, 115].

In [3]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print("GPU:", gpu.name)
    print("GPU Memory:", round(gpu.total_memory / 1024**3, 2), "GB")
else:
    print("No GPU available")

No GPU available


### Step 3: Create a Domain-Specific Dataset

**Question:**
> *How do we construct a custom question-and-answer dataset for domain adaptation (e.g., a university campus shop)[cite: 73, 115]?*

**Explanation:**
* **Domain Adaptation**: Pre-trained models understand general language[cite: 78] but lack specific internal knowledge about local services (like "Campus Essentials")[cite: 73].
* **Structured Q&A**: We define clean prompt-response pairs using the Hugging Face `Dataset` API[cite: 73, 99].

In [4]:
from datasets import Dataset

data = [
    {
        "question": "What products does Campus Essentials sell?",
        "answer": "Campus Essentials sells notebooks, pens, textbooks, USB drives, water bottles, and university merchandise."
    },
    {
        "question": "How much does a notebook cost?",
        "answer": "A standard notebook costs LKR 350."
    },
    {
        "question": "Do you sell computer science textbooks?",
        "answer": "Yes. Selected computer science textbooks are available, subject to stock availability."
    },
    {
        "question": "What time does the shop open?",
        "answer": "The shop opens at 8:30 AM from Monday to Friday."
    },
    {
        "question": "Can I return a damaged product?",
        "answer": "Yes. Customers can request an exchange for a damaged product by presenting the receipt within seven days of purchase."
    },
    {
        "question": "Do you sell USB drives?",
        "answer": "Yes. USB drives are available in selected storage capacities."
    },
    {
        "question": "Can I pay by card?",
        "answer": "Yes. The shop accepts cash and debit or credit cards."
    },
    {
        "question": "Are university-branded T-shirts available?",
        "answer": "Yes. University-branded T-shirts are available in selected sizes."
    },
    {
        "question": "Can I reserve a textbook?",
        "answer": "Yes. Customers can request a textbook reservation by contacting the shop. Reservations depend on stock availability."
    },
    {
        "question": "Where is Campus Essentials located?",
        "answer": "Campus Essentials is located beside the university library."
    }
]

print("Number of examples:", len(data))
dataset = Dataset.from_list(data)

C:\Users\ISIWARA KUMARAGE\Documents\NLP_Year_3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of examples: 10


### Step 4: Import Base Model and Tokenizer

**Question:**
> *How do we load an instruction-tuned causal language model (such as `Qwen/Qwen2.5-0.5B-Instruct`) and configure its tokenizer for CPU execution[cite: 73, 115]?*

**Explanation:**
* **Model Loading**: Imports `AutoModelForCausalLM` using standard 32-bit floating-point precision (`torch.float32`) since CUDA is not present[cite: 73].
* **Padding Token**: Ensures the tokenizer has a valid padding token configured by setting it equal to the end-of-sequence (`eos_token`)[cite: 73].

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Force float32 for CPU compatibility
dtype = torch.float32

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype
)

print("Model loaded successfully on CPU!")
print("Parameters:", f"{model.num_parameters():,}")

C:\Users\ISIWARA KUMARAGE\Documents\NLP_Year_3\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ISIWARA KUMARAGE\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
W1017 14:58:5

Model loaded successfully on CPU!
Parameters: 494,032,768


### Step 5: Construct Chat Template Formatting

**Question:**
> *How do we format raw text pairs into standardized model-specific chat templates for instruction tuning[cite: 73, 81]?*

**Explanation:**
* **Chat Templates**: Wraps user questions and assistant answers using role markers so the model learns conversational structure[cite: 73, 81].

In [6]:
def format_example(example):
    messages = [
        {
            "role": "user",
            "content": example["question"]
        },
        {
            "role": "assistant",
            "content": example["answer"]
        }
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False
        )
    }

train_dataset = dataset.map(
    format_example,
    remove_columns=dataset.column_names
)

print(train_dataset[0]["text"])

Map: 100%|██████████| 10/10 [00:00<00:00, 179.03 examples/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What products does Campus Essentials sell?<|im_end|>
<|im_start|>assistant
Campus Essentials sells notebooks, pens, textbooks, USB drives, water bottles, and university merchandise.<|im_end|>



### Step 6: Parameter-Efficient Fine-Tuning (LoRA) Configuration

**Question:**
> *How do we configure Low-Rank Adaptation (LoRA) to update only a tiny fraction of model weights[cite: 101, 102]?*

**Explanation:**
* **LoRA Setup**: Inserts low-rank decomposition matrices into attention projections, reducing the number of parameters trained[cite: 101, 102].

In [7]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


### Step 7: Train the Model Using SFTTrainer (CPU Version)

**Question:**
> *How do we execute supervised fine-tuning using Hugging Face's TRL library (`SFTTrainer`) on a CPU-only local environment[cite: 73, 98]?*

**Explanation:**
* **CPU Training Configuration**: We set `fp16=False` and use standard 32-bit floating-point precision so the training loop runs correctly on your CPU without throwing CUDA errors[cite: 73].
* **Execution**: `trainer.train()` updates our LoRA adapter weights using our custom campus shop dataset[cite: 73].

In [8]:
import torch
from trl import SFTTrainer, SFTConfig

print("Configuring SFTTrainer for CPU execution...")

training_args = SFTConfig(
    output_dir="./uni_fundamentals_lora",
    num_train_epochs=3,
    per_device_train_batch_size=1,  # Lower batch size for CPU efficiency
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=2,
    save_strategy="no",  # Skip saving checkpoints during quick CPU run to save time
    fp16=False,          # Must be False for CPU
    bf16=False,
    gradient_checkpointing=False,
    max_length=256,
    dataset_text_field="text",
    report_to="none",
    seed=42
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer
)

print("Starting training loop...")
trainer.train()
print("Training completed successfully!")

Configuring SFTTrainer for CPU execution...


Truncating train dataset: 100%|██████████| 10/10 [00:00<00:00, 1319.09 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 10/10 [00:00<00:00, 2583.34 examples/s]


Starting training loop...


C:\Users\ISIWARA KUMARAGE\Documents\NLP_Year_3\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
2,4.288291
4,2.603971
6,1.674944
8,1.359810
10,1.003438
12,0.950565
14,0.937897


Training completed successfully!


### Step 8: Save the Model Adapters

**Question:**
> *How do we serialize and save our trained LoRA adapter weights for future inference and deployment[cite: 73]?*

**Explanation:**
* **Adapter Persistence**: Saves only the lightweight LoRA parameter weights and tokenizer settings rather than the entire base model[cite: 73].

In [9]:
output_dir = "./uni_fundamentals_lora/final_adapter"

trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print("LoRA adapter saved successfully to:", output_dir)

LoRA adapter saved successfully to: ./uni_fundamentals_lora/final_adapter


### Step 9: Test Unseen Questions on Base vs. Fine-Tuned Models

**Question:**
> *How do we evaluate and compare responses from the original base model versus the fine-tuned model using questions absent from the training dataset[cite: 73]?*

**Explanation:**
* **Inference Pipeline**: Loads the base model alongside the saved LoRA adapter using `PeftModel.from_pretrained` and generates text responses for qualitative comparison[cite: 73].

In [10]:
import gc
from peft import PeftModel
from transformers import AutoModelForCausalLM

test_questions = [
   "Do you sell laptop chargers?",
   "Can I buy stationery and textbooks from the same shop?",
]

def generate_answer(model, question):
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# Clear memory and load fine-tuned model
if "model" in globals():
    del model
gc.collect()

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32
)

fine_tuned_model = PeftModel.from_pretrained(base_model, output_dir)

print("--- Testing Fine-Tuned Model ---")
for question in test_questions:
    answer = generate_answer(fine_tuned_model, question)
    print("\nQuestion:", question)
    print("Fine-tuned model:", answer)

Loading weights: 100%|██████████| 290/290 [00:02<00:00, 140.49it/s]


--- Testing Fine-Tuned Model ---

Question: Do you sell laptop chargers?
Fine-tuned model: Yes, we do sell laptop chargers.

Question: Can I buy stationery and textbooks from the same shop?
Fine-tuned model: Yes, you can buy stationery and textbooks from the same shop.
